# S03 · Split-KV decoding and the LSE merge (FlashDecoding, cascade attention)

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 10. Sources: github.com/Dao-AILab/flash-attention (25k stars, BSD-3); FlashInfer cascade (arXiv:2501.01005).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Show that attention over disjoint KV chunks merges exactly, and use the merge for cascade
(shared-prefix) attention.

## Theory and equations
$\text{LSE} = \log(e^{\text{LSE}_A} + e^{\text{LSE}_B})$, $o = e^{\text{LSE}_A-\text{LSE}} o_A + e^{\text{LSE}_B-\text{LSE}} o_B$.

## Diagram
```
KV:  [ chunk 0 ][ chunk 1 ][ chunk 2 ][ chunk 3 ]   -> processed in parallel
      (o0,l0)    (o1,l1)    (o2,l2)    (o3,l3)       -> merged pairwise, exactly
```

## Implementation

In [ ]:
import torch.nn.functional as F
from inference_lab import flash
torch.manual_seed(0)
q = torch.randn(1, 8, 1, 128)                      # one decode query, 8 heads
k, v = torch.randn(1, 8, 8192, 128), torch.randn(1, 8, 8192, 128)
ref = F.scaled_dot_product_attention(q, k, v)
for s in (1, 2, 8, 64):
    err = float((flash.split_kv_attention(q, k, v, s) - ref).abs().max())
    print(f"splits={s:3d}  max |split - SDPA| = {err:.2e}")

## Experiment: cascade attention for a shared prefix

In [ ]:
prefix_k, prefix_v = k[:, :, :6000], v[:, :, :6000]
Q = torch.randn(4, 8, 1, 128)                      # 4 users, same prefix
o_pre, lse_pre = flash.partial_state(Q, prefix_k.expand(4, -1, -1, -1), prefix_v.expand(4, -1, -1, -1))
for i in range(4):
    ks, vs = torch.randn(1, 8, 300, 128), torch.randn(1, 8, 300, 128)
    o_suf, lse_suf = flash.partial_state(Q[i:i+1], ks, vs)
    o, _ = flash.merge_states(o_pre[i:i+1], lse_pre[i:i+1], o_suf, lse_suf)
    full = F.scaled_dot_product_attention(Q[i:i+1], torch.cat([prefix_k, ks], 2), torch.cat([prefix_v, vs], 2))
    print(f"user {i}: max |cascade - full| = {float((o - full).abs().max()):.1e}")

## Interpretation
Any split is exact. Cascade attention reads the shared prefix once per batch instead of once per
request, which is the bandwidth saving FlashInfer's cascade kernels exploit.

## Limitations
PyTorch loops; no parallel hardware scheduling.

## Conclusion
One identity, the LSE merge, underlies split-KV decoding, cascade attention and context parallelism.